# Notebook 07 — Sentiment and emotion analysis

**Project:** Study 2 — Chatbot Neuroticism (DGK Lab)  
**Pipeline stage:** 7 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~10-15 minutes (GPU recommended for transformer inference)  
**API costs:** $0 (all local models)

## Purpose

Apply three independent, non-LLM sentiment/affect methods to the 486 substantive OE responses. The goal is to bring in complementary continuous signals on emotional valence and discrete emotion categories that the rubric-coded affect codes cannot fully capture on their own.

**Three methods, three perspectives:**

1. **Roberta sentiment** (`cardiffnlp/twitter-roberta-base-sentiment-latest`): transformer-based 3-class (positive / neutral / negative) probabilities. Modern semantic model.
2. **DistilRoberta emotion** (`j-hartmann/emotion-english-distilroberta-base`): transformer-based 7-class (anger / disgust / fear / joy / neutral / sadness / surprise) probabilities. Ekman's 6 basic emotions + neutral.
3. **VADER** (`vaderSentiment.SentimentIntensityAnalyzer`): lexicon-based compound score in [-1, +1]. Established psychometric tool, robust on short text.

## Why three methods

Sentiment analysis is methodologically contested. Transformer models capture semantic context but were trained on different corpora (Twitter for the Roberta model — short, informal). Lexicon-based methods like VADER have transparent rules and well-validated psychometric properties but miss negation and sarcasm.

By reporting three independent methods, the manuscript can claim convergent evidence where they agree and surface methodological caveats where they don't.

## What this notebook does NOT do

- **No LLM-based sentiment scoring** (would duplicate the rubric-coded affect codes from NB04/05)
- **No emotion-only clustering** (would produce degenerate clusters that just recapitulate emotion labels — see Notebook 06 for the principled topic modeling on text embeddings)
- **No hypothesis testing**. Descriptive only. Rmd does inference.
- **No sentiment-by-sentence breakdowns.** Each OE response is one analytic unit.

## What this notebook DOES do

1. Score each response with all three methods → `sentiment_per_response.parquet`
2. Aggregate to participant level → `sentiment_per_participant.csv`
3. Compute pairwise inter-method agreement → `sentiment_inter_model_agreement.csv`
4. Correlate continuous sentiment with rubric valence codes (convergent validity) → `sentiment_rubric_convergence.csv`
5. Characterize the 12 topics from Notebook 06 by their emotion profiles → `topic_emotion_alignment.csv`

## Inputs

- `data/derived/open_ends_long.parquet` (from Notebook 01)
- `data/derived/coding/consensus_long.parquet` (from Notebook 05 — for convergent validity)
- `data/derived/topics/topics_per_response.parquet` (from Notebook 06 — for topic-emotion alignment)

## Outputs

- `data/derived/sentiment/sentiment_per_response.parquet`
- `data/derived/sentiment/sentiment_per_participant.csv`
- `data/derived/sentiment/sentiment_inter_model_agreement.csv`
- `data/derived/sentiment/sentiment_rubric_convergence.csv`
- `data/derived/sentiment/topic_emotion_alignment.csv`
- `data/derived/sentiment/sentiment_manifest.json`

## Methodological caveats (for Methods writeup)

- Transformer sentiment models were trained on corpora (Twitter) different from our reflective-essay-like OE responses. Predictions on **short responses** (< 5 words) are unreliable; we report confidence per prediction.
- The 7-class emotion model uses Ekman's basic emotions, not dimensional affect (e.g., PANAS). These are categorical attributions, not continuous valence/arousal coordinates.
- All three methods give per-response outputs. Aggregation to participant level (mean across the four OEs) is descriptively informative but conflates question content.

## 1. Environment setup

In [ ]:
import sys, os, json, time, hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any

# Install transformers + VADER if needed
def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_ensure('transformers')
_ensure('torch')
_ensure('vaderSentiment')

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'pandas:       {pd.__version__}')
print(f'numpy:        {np.__version__}')
print(f'torch:        {torch.__version__}')
print(f'Compute device: {device}')
if device == 'cpu':
    print(f'  [INFO] No GPU detected. Transformer inference will be slower (~30-60 min total).')
    print(f'         If on Colab, set Runtime > Change runtime type > T4 GPU for ~5-10 min.')


## 2. Project paths

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 2')
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_DERIVED = PROJECT_ROOT / 'data' / 'derived'
SENTIMENT_DIR = DATA_DERIVED / 'sentiment'
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)

PATH_OES        = DATA_DERIVED / 'open_ends_long.parquet'
PATH_CONSENSUS  = DATA_DERIVED / 'coding' / 'consensus_long.parquet'
PATH_TOPICS     = DATA_DERIVED / 'topics' / 'topics_per_response.parquet'

for p in [PATH_OES, PATH_CONSENSUS, PATH_TOPICS]:
    if not p.exists():
        raise FileNotFoundError(f'Missing input: {p}')

print(f'Project root:  {PROJECT_ROOT}')
print(f'Sentiment dir: {SENTIMENT_DIR}')


## 3. Configuration

In [ ]:
# Pinned model revisions for reproducibility
ROBERTA_SENTIMENT_MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
ROBERTA_SENTIMENT_REVISION = None  # use latest stable

EMOTION_MODEL = 'j-hartmann/emotion-english-distilroberta-base'
EMOTION_REVISION = None  # use latest stable

# Class labels (in order returned by these specific Hugging Face models)
ROBERTA_SENTIMENT_LABELS = ['negative', 'neutral', 'positive']
EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# OE fields to score (all four primary OEs)
TARGET_QUESTIONS = ['oe_experience', 'oe_person_or_tool', 'oe_responsibility', 'oe_compensation']

# Minimum word count: we score everything but flag low-confidence outputs for short responses
MIN_WORDS_FOR_RELIABLE_SENTIMENT = 5

# Batch size for transformer inference
BATCH_SIZE = 16

print(f'Sentiment model: {ROBERTA_SENTIMENT_MODEL}')
print(f'Emotion model:   {EMOTION_MODEL}')
print(f'Batch size:      {BATCH_SIZE}')
print(f'Questions:       {TARGET_QUESTIONS}')


## 4. Load OE data, rubric codes, and topic assignments

In [ ]:
oe = pd.read_parquet(PATH_OES)
print(f'OE long-form: {oe.shape}')

# Filter to substantive responses for primary OE questions (>= 3 words, same as Notebook 04/06)
oe_filtered = oe[
    oe['oe_field'].isin(TARGET_QUESTIONS) &
    ~oe['is_blank'] &
    (oe['n_words'] >= 3)
].copy().reset_index(drop=True)
print(f'After filter: {len(oe_filtered)} responses')
print(f'  per question: {oe_filtered["oe_field"].value_counts().to_dict()}')
print(f'  word count: min={oe_filtered["n_words"].min()}, median={oe_filtered["n_words"].median():.0f}, max={oe_filtered["n_words"].max()}')
short = (oe_filtered['n_words'] < MIN_WORDS_FOR_RELIABLE_SENTIMENT).sum()
print(f'  short responses (<{MIN_WORDS_FOR_RELIABLE_SENTIMENT} words): {short} will be flagged low-confidence')

# Load consensus codes (for convergent validity)
consensus = pd.read_parquet(PATH_CONSENSUS)
print(f'\nConsensus codes: {consensus.shape}')

# Load topic assignments (for topic-emotion alignment)
topics = pd.read_parquet(PATH_TOPICS)
print(f'Topic assignments: {topics.shape}')


## 5. Roberta sentiment (3-class)

Apply `cardiffnlp/twitter-roberta-base-sentiment-latest` to each response. Returns per-class probabilities for negative, neutral, positive.

Note: this model was trained on Twitter data. Its predictions on our reflective-essay-like text should be interpreted with appropriate uncertainty.


In [ ]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def run_classifier_batch(model_name, texts, labels, batch_size=BATCH_SIZE):
    """Run a HuggingFace sequence-classification model on a list of texts, batched."""
    print(f'  Loading {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()

    all_probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=512,
                            return_tensors='pt').to(device)
            logits = model(**enc).logits.cpu().numpy()
            probs = softmax(logits)
            all_probs.append(probs)
            if (i // batch_size) % 10 == 0:
                print(f'    batch {i//batch_size+1}/{(len(texts)-1)//batch_size+1}')

    probs_arr = np.vstack(all_probs)
    return probs_arr

# Score all responses with Roberta sentiment
print('\n=== Roberta sentiment ===')
t0 = time.time()
roberta_probs = run_classifier_batch(ROBERTA_SENTIMENT_MODEL,
                                       oe_filtered['response'].tolist(),
                                       ROBERTA_SENTIMENT_LABELS)
print(f'  Done in {time.time()-t0:.1f}s. Shape: {roberta_probs.shape}')

# Add to dataframe
for j, lab in enumerate(ROBERTA_SENTIMENT_LABELS):
    oe_filtered[f'roberta_{lab}'] = roberta_probs[:, j]

# Argmax label and confidence
oe_filtered['roberta_label'] = [ROBERTA_SENTIMENT_LABELS[i] for i in roberta_probs.argmax(axis=1)]
oe_filtered['roberta_confidence'] = roberta_probs.max(axis=1)

# Continuous score: P(positive) - P(negative), range [-1, +1]
oe_filtered['roberta_score'] = oe_filtered['roberta_positive'] - oe_filtered['roberta_negative']

# Free GPU memory after each large model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\nRoberta sentiment distribution:')
print(oe_filtered['roberta_label'].value_counts())
print(f'\nMean confidence: {oe_filtered["roberta_confidence"].mean():.3f}')


## 6. DistilRoberta emotion (7-class)

In [ ]:
print('=== DistilRoberta emotion ===')
t0 = time.time()
emotion_probs = run_classifier_batch(EMOTION_MODEL,
                                       oe_filtered['response'].tolist(),
                                       EMOTION_LABELS)
print(f'  Done in {time.time()-t0:.1f}s. Shape: {emotion_probs.shape}')

for j, lab in enumerate(EMOTION_LABELS):
    oe_filtered[f'emotion_{lab}'] = emotion_probs[:, j]

oe_filtered['emotion_label'] = [EMOTION_LABELS[i] for i in emotion_probs.argmax(axis=1)]
oe_filtered['emotion_confidence'] = emotion_probs.max(axis=1)

torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\nEmotion label distribution:')
print(oe_filtered['emotion_label'].value_counts())
print(f'\nMean confidence: {oe_filtered["emotion_confidence"].mean():.3f}')


## 7. VADER (lexicon-based)

VADER's compound score is a normalized weighted composite in [-1, +1]. Conventional cutoffs:
- compound ≥ +0.05 → positive
- −0.05 < compound < +0.05 → neutral
- compound ≤ −0.05 → negative

Continuous score reported as well as the categorical label.


In [ ]:
vader = SentimentIntensityAnalyzer()

print('=== VADER ===')
t0 = time.time()
scores = [vader.polarity_scores(t) for t in oe_filtered['response'].tolist()]
oe_filtered['vader_pos']      = [s['pos'] for s in scores]
oe_filtered['vader_neg']      = [s['neg'] for s in scores]
oe_filtered['vader_neu']      = [s['neu'] for s in scores]
oe_filtered['vader_compound'] = [s['compound'] for s in scores]
print(f'  Done in {time.time()-t0:.2f}s.')

# Categorical label from compound score
def vader_label(c):
    if c >= 0.05: return 'positive'
    elif c <= -0.05: return 'negative'
    else: return 'neutral'

oe_filtered['vader_label'] = oe_filtered['vader_compound'].apply(vader_label)

print(f'\nVADER label distribution:')
print(oe_filtered['vader_label'].value_counts())
print(f'\nVADER compound score: mean={oe_filtered["vader_compound"].mean():.3f}, '
      f'sd={oe_filtered["vader_compound"].std():.3f}')


## 8. Per-response sentiment table

In [ ]:
# Add a low-confidence flag for short responses
oe_filtered['is_short_response'] = oe_filtered['n_words'] < MIN_WORDS_FOR_RELIABLE_SENTIMENT

# Column order
keep_cols = (
    ['sona_id', 'condition_label', 'oe_field', 'response', 'n_words', 'is_short_response']
    + [f'roberta_{lab}' for lab in ROBERTA_SENTIMENT_LABELS]
    + ['roberta_label', 'roberta_confidence', 'roberta_score']
    + [f'emotion_{lab}' for lab in EMOTION_LABELS]
    + ['emotion_label', 'emotion_confidence']
    + ['vader_pos', 'vader_neg', 'vader_neu', 'vader_compound', 'vader_label']
)
sentiment_long = oe_filtered[keep_cols].copy()

print(f'Sentiment per-response: {sentiment_long.shape}')
print(f'  Columns ({len(sentiment_long.columns)}): {list(sentiment_long.columns)}')
print(f'\nSample rows:')
print(sentiment_long[['sona_id', 'oe_field', 'roberta_label', 'emotion_label',
                        'vader_label', 'vader_compound']].head(8).to_string(index=False))


## 9. Per-participant aggregation

In [ ]:
# Aggregate to participant level: mean across the four OEs for continuous scores
# We compute (a) overall mean (across all 4 OEs) and (b) per-OE means

# Overall mean (one row per sona_id)
participant_overall = sentiment_long.groupby(['sona_id', 'condition_label']).agg(
    n_oe_responses=('response', 'count'),
    roberta_score_mean=('roberta_score', 'mean'),
    roberta_positive_mean=('roberta_positive', 'mean'),
    roberta_negative_mean=('roberta_negative', 'mean'),
    emotion_joy_mean=('emotion_joy', 'mean'),
    emotion_sadness_mean=('emotion_sadness', 'mean'),
    emotion_neutral_mean=('emotion_neutral', 'mean'),
    vader_compound_mean=('vader_compound', 'mean'),
).reset_index()

# Per-OE wide form (continuous compound, one column per OE)
per_oe_wide = sentiment_long.pivot_table(
    index=['sona_id', 'condition_label'],
    columns='oe_field',
    values='vader_compound',
    aggfunc='first',
).reset_index()
per_oe_wide.columns = [f'vader_compound_{c}' if c not in ('sona_id', 'condition_label') else c
                        for c in per_oe_wide.columns]

# Merge
participant_df = participant_overall.merge(per_oe_wide, on=['sona_id', 'condition_label'], how='left')

print(f'Per-participant: {participant_df.shape}')
print(f'  Conditions: {participant_df["condition_label"].value_counts().to_dict()}')
print(f'\nDescriptive: mean continuous scores by condition (descriptive only — no testing):')
print(participant_df.groupby('condition_label').agg(
    n=('sona_id', 'count'),
    roberta_score=('roberta_score_mean', 'mean'),
    vader_compound=('vader_compound_mean', 'mean'),
    emotion_joy=('emotion_joy_mean', 'mean'),
    emotion_sadness=('emotion_sadness_mean', 'mean'),
).round(3).to_string())


## 10. Inter-model agreement

Pairwise correlations between the three continuous sentiment scores:
- roberta_score (P(pos) - P(neg), range [-1, +1])
- vader_compound (-1 to +1)
- emotion_joy - emotion_sadness (proxy for valence in the emotion model)

These are not the same construct, but they should correlate if all three are picking up emotional valence.


In [ ]:
sentiment_long['emotion_joy_minus_sad'] = (
    sentiment_long['emotion_joy'] - sentiment_long['emotion_sadness']
)

agreement_cols = ['roberta_score', 'vader_compound', 'emotion_joy_minus_sad']
labels = {'roberta_score': 'Roberta', 'vader_compound': 'VADER',
          'emotion_joy_minus_sad': 'Emotion (joy-sad)'}

print('Pairwise Pearson correlations between continuous sentiment scores:\n')
print(f'  (all {len(sentiment_long)} responses)\n')
agreement_rows = []
for i, a in enumerate(agreement_cols):
    for b in agreement_cols[i+1:]:
        r = sentiment_long[[a, b]].corr().iloc[0, 1]
        agreement_rows.append({'method_a': labels[a], 'method_b': labels[b],
                                'pearson_r': round(r, 3), 'scope': 'all_responses'})
        print(f'  {labels[a]:<18s} vs {labels[b]:<22s} r = {r:+.3f}')

# Also compute on substantive-only (n_words >= 5)
print('\n  (substantive responses only, n_words >= 5)\n')
sub_subset = sentiment_long[~sentiment_long['is_short_response']]
for i, a in enumerate(agreement_cols):
    for b in agreement_cols[i+1:]:
        r = sub_subset[[a, b]].corr().iloc[0, 1]
        agreement_rows.append({'method_a': labels[a], 'method_b': labels[b],
                                'pearson_r': round(r, 3), 'scope': 'substantive_only',
                                'n_responses': len(sub_subset)})
        print(f'  {labels[a]:<18s} vs {labels[b]:<22s} r = {r:+.3f}')

agreement_df = pd.DataFrame(agreement_rows)


## 11. Convergent validity with rubric codes

Correlate continuous sentiment scores (Roberta, VADER, emotion joy/sadness)
with rubric-coded valence indicators on OE1 (oe_experience).

**Composite valence indicators.** The current rubric (NB03 v2) does not have
a single `overall_positive_valence` / `overall_negative_valence` code as the
previous rubric version did. Instead, several finer-grained affect codes
capture each side of valence. We build **composite indicators**:

- `composite_positive_valence` = 1 if any of `brief_generic_positive`,
  `felt_validated_supported`, or `emotionally_moved` fired for that
  participant.
- `composite_negative_valence` = 1 if any of `annoyed_or_bored`,
  `uncomfortable_or_uneasy`, `felt_unheard_or_misdirected`, or
  `sylva_too_self_focused` fired.

This composite approach is more robust to rubric regenerations than tying
convergent validity to a single binary code, and aggregates the
finer-grained codes into broad positive/negative bins.

**Expectation.** If the three sentiment methods are valid measures of OE1
emotional valence, we should see:
- Positive Pearson r between each method and `composite_positive_valence`
- Negative r between Roberta/VADER and `composite_negative_valence`
(Emotion: joy / sadness are themselves discrete-emotion probabilities, not
valence per se, so their correlations with the composites are
informational, not validity checks.)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Convergent validity: continuous sentiment × rubric composite valence codes
# ─────────────────────────────────────────────────────────────────────────────
# The rubric was regenerated in NB03 and the prior codes
# `overall_positive_valence` / `overall_negative_valence` no longer exist.
# Build composite valence indicators from the new rubric's affect codes.

# Restrict sentiment data to OE1 (where affect codes are defined)
sent_oe1 = sentiment_long[sentiment_long['oe_field'] == 'oe_experience'].copy()

# Build the two valence composites from the ACTUAL oe_experience codes NB03 produced
# (verified against reliability_table.csv categories). Positive = the rubric's
# 'positive_attributes' category; negative = its 'critiques' category. If NB03 is
# re-run and the rubric drifts, the WARNING below flags any code that goes missing.
POSITIVE_AFFECT_CODES = [
    'empathetic_caring',          # praised Sylva as warm/caring (positive_attributes)
    'immersion_engagement',       # enjoyed / felt immersed (positive_attributes)
]
NEGATIVE_AFFECT_CODES = [
    'surface_level_lack_depth',   # critique: shallow / lacked depth
    'repetitive_format',          # critique: repetitive
    'sylva_self_focused',         # critique: Sylva too self-focused
    'roleplay_friction',          # critique: roleplay felt off
    'sycophantic_agreement',      # critique: agreed too readily
    'concern_about_harm',         # critique: raised harm concerns
]
ALL_AFFECT_CODES = POSITIVE_AFFECT_CODES + NEGATIVE_AFFECT_CODES

# Pull these codes from consensus_long for OE1
cons_val = consensus[
    (consensus['oe_field'] == 'oe_experience') &
    (consensus['code_id'].isin(ALL_AFFECT_CODES))
][['sona_id', 'code_id', 'consensus']].copy()
cons_val['consensus'] = pd.to_numeric(cons_val['consensus'], errors='coerce').fillna(0).astype(int)

# Warn loudly if the rubric has shifted again (some codes missing)
found_codes = set(cons_val['code_id'].unique())
missing = sorted(set(ALL_AFFECT_CODES) - found_codes)
if missing:
    print('\u26a0  WARNING: the following affect codes are missing from consensus_long:')
    for c in missing: print(f'     - {c}')
    print('   The rubric may have shifted. Inspect data/derived/rubric/rubric_v2.md')
    print('   and update POSITIVE_AFFECT_CODES / NEGATIVE_AFFECT_CODES above.')
else:
    print(f'\u2713  All {len(ALL_AFFECT_CODES)} expected affect codes found in consensus_long.')

# Pivot to wide (one column per code, 0/1 consensus)
val_wide = cons_val.pivot_table(
    index='sona_id', columns='code_id', values='consensus', aggfunc='first'
).reset_index()

# Build the two composite indicators. ANY constituent code firing -> composite = 1.
present_pos = [c for c in POSITIVE_AFFECT_CODES if c in val_wide.columns]
present_neg = [c for c in NEGATIVE_AFFECT_CODES if c in val_wide.columns]
val_wide['composite_positive_valence'] = (
    val_wide[present_pos].fillna(0).sum(axis=1).clip(upper=1).astype(int)
    if present_pos else 0
)
val_wide['composite_negative_valence'] = (
    val_wide[present_neg].fillna(0).sum(axis=1).clip(upper=1).astype(int)
    if present_neg else 0
)

# Merge with OE1 sentiment
merged = sent_oe1.merge(val_wide, on='sona_id', how='left')
for c in ['composite_positive_valence', 'composite_negative_valence']:
    if c not in merged.columns:
        merged[c] = 0
    merged[c] = merged[c].fillna(0).astype(int)

print(f'\nMerged OE1 + composites: {merged.shape}')
print(f'  composite_positive_valence prevalence: {merged["composite_positive_valence"].mean():.3f}')
print(f'  composite_negative_valence prevalence: {merged["composite_negative_valence"].mean():.3f}')

# Point-biserial correlations (binary composite \u00d7 continuous sentiment)
VALENCE_COMPOSITES = ['composite_positive_valence', 'composite_negative_valence']
convergence_rows = []
for method_col, method_label in [
    ('roberta_score',    'Roberta'),
    ('vader_compound',   'VADER'),
    ('emotion_joy',      'Emotion: joy'),
    ('emotion_sadness',  'Emotion: sadness'),
]:
    for code in VALENCE_COMPOSITES:
        r = merged[[method_col, code]].corr().iloc[0, 1]
        convergence_rows.append({
            'method': method_label,
            'rubric_code': code,
            'pearson_r': round(r, 3),
        })

convergence_df = pd.DataFrame(convergence_rows)
print('\nConvergent validity (point-biserial r on OE1, composite codes):\n')
print(convergence_df.pivot_table(index='method', columns='rubric_code',
                                 values='pearson_r').to_string())
print('\nInterpretation:')
print('  - Roberta and VADER should correlate POSITIVELY with composite_positive_valence')
print('    and NEGATIVELY with composite_negative_valence.')
print('  - Emotion:joy and Emotion:sadness are discrete-emotion probabilities, not')
print('    valence per se; their r values are informational rather than validity checks.')
print(f'\nConstituent codes used:')
print(f'  positive: {present_pos}')
print(f'  negative: {present_neg}')

## 12. Emotion characterization of topics

For each topic from Notebook 06 (primary mpnet), compute the mean emotion profile across its member responses. Identify which emotions are over-represented in each topic vs the question's overall emotion distribution.

This **does not cluster on emotion vectors** (which would be degenerate). Instead, it characterizes existing topic clusters by their emotional signature, paralleling the topic-rubric alignment from Notebook 06.


In [ ]:
# Merge topic assignments with emotion probabilities
# topics has sona_id, oe_field, topic_id, opus_label
# sentiment_long has sona_id, oe_field, emotion_* columns

emotion_prob_cols = [f'emotion_{lab}' for lab in EMOTION_LABELS]
topic_sentiment = topics.merge(
    sentiment_long[['sona_id', 'oe_field'] + emotion_prob_cols],
    on=['sona_id', 'oe_field'],
    how='inner'
)
print(f'Topics × sentiment merge: {topic_sentiment.shape}')

# Per (oe_field × topic_id), compute mean emotion vector AND compare to question average
alignment_rows = []
for q in TARGET_QUESTIONS:
    q_data = topic_sentiment[topic_sentiment['oe_field'] == q]
    if len(q_data) == 0: continue
    q_mean_emotion = q_data[emotion_prob_cols].mean()

    for tid in sorted(q_data['topic_id'].unique()):
        topic_data = q_data[q_data['topic_id'] == tid]
        topic_mean = topic_data[emotion_prob_cols].mean()
        topic_label = topic_data['opus_label'].iloc[0]
        n_docs = len(topic_data)

        for lab in EMOTION_LABELS:
            col = f'emotion_{lab}'
            alignment_rows.append({
                'oe_field':     q,
                'topic_id':     int(tid),
                'opus_label':   topic_label,
                'emotion':      lab,
                'n_docs':       n_docs,
                'p_topic':      round(topic_mean[col], 4),
                'p_question':   round(q_mean_emotion[col], 4),
                'diff':         round(topic_mean[col] - q_mean_emotion[col], 4),
            })

emotion_alignment_df = pd.DataFrame(alignment_rows)

# Show top emotion residual per topic
print('\nDominant emotion residual per topic (top emotion over-represented vs question mean):\n')
for q in TARGET_QUESTIONS:
    df_q = emotion_alignment_df[emotion_alignment_df['oe_field'] == q]
    if len(df_q) == 0: continue
    print(f'\n--- {q} ---')
    for tid in sorted(df_q['topic_id'].unique()):
        topic_df = df_q[df_q['topic_id'] == tid].sort_values('diff', ascending=False)
        if topic_df.empty: continue
        label = topic_df['opus_label'].iloc[0]
        n = topic_df['n_docs'].iloc[0]
        top_emotion = topic_df.iloc[0]
        print(f'  Topic {int(tid):>2} [{label}] (n={n}): '
              f'most over-represented emotion = {top_emotion["emotion"]} '
              f'(p_topic={top_emotion["p_topic"]:.3f} vs question {top_emotion["p_question"]:.3f}, '
              f'Δ={top_emotion["diff"]:+.3f})')


## 13. Sanity checks

In [ ]:
checks = []

# (1) Every input OE response got scored by all three methods
n_scored = sentiment_long[['roberta_label', 'emotion_label', 'vader_label']].notna().all(axis=1).sum()
checks.append(('All input responses scored by all 3 methods',
               n_scored == len(oe_filtered),
               f'{n_scored} of {len(oe_filtered)}'))

# (2) Probability rows sum to 1 (per method)
roberta_sums = sentiment_long[[f'roberta_{lab}' for lab in ROBERTA_SENTIMENT_LABELS]].sum(axis=1)
checks.append(('Roberta probability rows sum to ~1.0',
               (np.abs(roberta_sums - 1.0) < 0.01).all(),
               f'min={roberta_sums.min():.4f}, max={roberta_sums.max():.4f}'))

emotion_sums = sentiment_long[[f'emotion_{lab}' for lab in EMOTION_LABELS]].sum(axis=1)
checks.append(('Emotion probability rows sum to ~1.0',
               (np.abs(emotion_sums - 1.0) < 0.01).all(),
               f'min={emotion_sums.min():.4f}, max={emotion_sums.max():.4f}'))

# (3) Inter-model agreement is positive (all methods picking up some valence signal)
agreement_subset = agreement_df[agreement_df['scope'] == 'substantive_only']
all_positive = (agreement_subset['pearson_r'] > 0).all()
checks.append(('All inter-method correlations positive',
               all_positive,
               f'min r = {agreement_subset["pearson_r"].min():.3f}'))

# (4) Convergent validity: positive_valence should positively correlate with sentiment
pos_val_rows = convergence_df[convergence_df['rubric_code'] == 'composite_positive_valence']
sentiment_methods_with_positive_r = (pos_val_rows[pos_val_rows['method'].isin(['Roberta', 'VADER', 'Emotion: joy'])]
                                       ['pearson_r'] > 0).sum()
checks.append(('Roberta/VADER/joy positively correlate with composite_positive_valence',
               sentiment_methods_with_positive_r == 3,
               f'{sentiment_methods_with_positive_r} of 3 positive'))

# (5) Negative valence convergent
neg_val_rows = convergence_df[convergence_df['rubric_code'] == 'composite_negative_valence']
sentiment_methods_with_negative_r = (neg_val_rows[neg_val_rows['method'].isin(['Roberta', 'VADER'])]
                                       ['pearson_r'] < 0).sum()
checks.append(('Roberta/VADER negatively correlate with composite_negative_valence',
               sentiment_methods_with_negative_r == 2,
               f'{sentiment_methods_with_negative_r} of 2 negative'))

# (6) Topic-emotion alignment table covers every topic
n_topics = topics['topic_id'].nunique()
n_alignment_topics = emotion_alignment_df.groupby(['oe_field', 'topic_id']).ngroups
checks.append(('Emotion alignment covers all topics',
               n_alignment_topics == topic_sentiment.groupby(['oe_field', 'topic_id']).ngroups,
               f'aligned={n_alignment_topics}'))

# (7) Confidence quality: at least 75% of substantive predictions should be high confidence
mean_confidence = sentiment_long[~sentiment_long['is_short_response']][['roberta_confidence', 'emotion_confidence']].mean()
checks.append(('Mean Roberta confidence > 0.6 on substantive responses',
               mean_confidence['roberta_confidence'] > 0.6,
               f'mean={mean_confidence["roberta_confidence"]:.3f}'))

print('-' * 70)
all_pass = True
for name, passed, detail in checks:
    marker = '[OK]' if passed else '[FAIL]'
    if not passed: all_pass = False
    print(f'  {marker:<8s} {name}')
    if not passed:
        print(f'           -> {detail}')
print('-' * 70)
print('All checks passed.' if all_pass else 'SANITY CHECKS FAILED — review above.')

## 14. Write outputs

In [ ]:
# Per-response sentiment
sent_long_path = SENTIMENT_DIR / 'sentiment_per_response.parquet'
sentiment_long.to_parquet(sent_long_path, index=False)

# Per-participant aggregation
part_path = SENTIMENT_DIR / 'sentiment_per_participant.csv'
participant_df.to_csv(part_path, index=False)

# Inter-model agreement
agreement_path = SENTIMENT_DIR / 'sentiment_inter_model_agreement.csv'
agreement_df.to_csv(agreement_path, index=False)

# Convergent validity
convergence_path = SENTIMENT_DIR / 'sentiment_rubric_convergence.csv'
convergence_df.to_csv(convergence_path, index=False)

# Topic-emotion alignment
emotion_align_path = SENTIMENT_DIR / 'topic_emotion_alignment.csv'
emotion_alignment_df.to_csv(emotion_align_path, index=False)

# Helper: convert numpy scalar types to native Python primitives for JSON
def _py(x):
    """Convert numpy/pandas scalar to native Python type for JSON serialization."""
    if hasattr(x, 'item'):  # numpy/pandas scalars all have .item()
        return x.item()
    return x

# Manifest
manifest = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'models': {
        'roberta_sentiment': ROBERTA_SENTIMENT_MODEL,
        'emotion': EMOTION_MODEL,
        'vader': 'vaderSentiment.SentimentIntensityAnalyzer',
    },
    'n_responses_scored': int(len(sentiment_long)),
    'n_participants': int(participant_df['sona_id'].nunique()),
    'compute_device': str(device),
    'short_response_threshold_words': int(MIN_WORDS_FOR_RELIABLE_SENTIMENT),
    'n_short_responses_flagged': int(sentiment_long['is_short_response'].sum()),
    'descriptive_summary_by_condition': {
        cond: {
            'n': int((participant_df['condition_label'] == cond).sum()),
            'roberta_score_mean':  round(_py(participant_df[participant_df['condition_label'] == cond]['roberta_score_mean'].mean()), 4),
            'vader_compound_mean': round(_py(participant_df[participant_df['condition_label'] == cond]['vader_compound_mean'].mean()), 4),
            'emotion_joy_mean':    round(_py(participant_df[participant_df['condition_label'] == cond]['emotion_joy_mean'].mean()), 4),
            'emotion_sadness_mean': round(_py(participant_df[participant_df['condition_label'] == cond]['emotion_sadness_mean'].mean()), 4),
        } for cond in ['HighN', 'LowN']
    },
    'outputs': {
        'sentiment_per_response':         str(sent_long_path.relative_to(PROJECT_ROOT)),
        'sentiment_per_participant':      str(part_path.relative_to(PROJECT_ROOT)),
        'sentiment_inter_model_agreement': str(agreement_path.relative_to(PROJECT_ROOT)),
        'sentiment_rubric_convergence':   str(convergence_path.relative_to(PROJECT_ROOT)),
        'topic_emotion_alignment':        str(emotion_align_path.relative_to(PROJECT_ROOT)),
    },
}
manifest_path = SENTIMENT_DIR / 'sentiment_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=_py)

print(f'Wrote:')
print(f'  {sent_long_path}')
print(f'  {part_path}')
print(f'  {agreement_path}')
print(f'  {convergence_path}')
print(f'  {emotion_align_path}')
print(f'  {manifest_path}')


## 15. Wrap-up

In [ ]:
print('=' * 70)
print('NOTEBOOK 07 COMPLETE')
print('=' * 70)
print(f'Responses scored:    {len(sentiment_long)}')
print(f'Participants:        {participant_df["sona_id"].nunique()}')
print(f'Methods applied:     Roberta sentiment + DistilRoberta emotion + VADER')
print(f'Compute device:      {device}')
print()
print(f'Inter-method agreement (substantive responses, n_words \u22655):')
sub_agree = agreement_df[agreement_df["scope"] == "substantive_only"]
for _, row in sub_agree.iterrows():
    print(f'  {row["method_a"]:<18s} vs {row["method_b"]:<22s} r = {row["pearson_r"]:+.3f}')
print()
print(f'Convergent validity with rubric valence codes (OE1):')
print(f'  Roberta vs composite_positive_valence: r = '
      f'{convergence_df[(convergence_df["method"]=="Roberta") & (convergence_df["rubric_code"]=="composite_positive_valence")]["pearson_r"].iloc[0]:+.3f}')
print(f'  VADER vs composite_positive_valence:   r = '
      f'{convergence_df[(convergence_df["method"]=="VADER") & (convergence_df["rubric_code"]=="composite_positive_valence")]["pearson_r"].iloc[0]:+.3f}')
print(f'  Roberta vs composite_negative_valence: r = '
      f'{convergence_df[(convergence_df["method"]=="Roberta") & (convergence_df["rubric_code"]=="composite_negative_valence")]["pearson_r"].iloc[0]:+.3f}')
print(f'  VADER vs composite_negative_valence:   r = '
      f'{convergence_df[(convergence_df["method"]=="VADER") & (convergence_df["rubric_code"]=="composite_negative_valence")]["pearson_r"].iloc[0]:+.3f}')
print()
print(f'Topic-emotion alignment: {len(emotion_alignment_df)} (topic \u00d7 emotion) rows')
print()
print(f'Next: Notebook 08 (turn-level chat-log coding) — the largest remaining piece.')
print(f'      Then study2_nlp_integration.Rmd brings everything together.')